In [ ]:
import os
import json
import glob
import math
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------
# Config
# ----------------------------
RESULT_DIR = "./result"
OUT_DIR = "./analysis_out/ch5_1"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")

EVENT_ORDER = ["Continue", "Stable", "SizeChange", "Merge", "Split"]  # tolerate older label "Stable"
W_ORDER = [1, 2, 5, 10, 20, 50]


# ----------------------------
# Helpers
# ----------------------------
def ensure_dirs():
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(TAB_DIR, exist_ok=True)


def load_all_results(result_dir: str) -> Tuple[pd.DataFrame, List[dict]]:
    """
    Load all json result files in result_dir.
    Each file is expected to be a list of trial dicts.
    Returns: (df_trials, raw_trials)
    """
    paths = sorted(glob.glob(os.path.join(result_dir, "*.json")))
    if not paths:
        raise FileNotFoundError(f"No JSON files found in {result_dir}")

    raw_trials: List[dict] = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise ValueError(f"Result file must contain a list of trials: {p}")
        for t in data:
            t["_source_file"] = os.path.basename(p)
            raw_trials.append(t)

    # flatten
    rows = []
    for t in raw_trials:
        gt = t.get("ground_truth")
        # Backward compatibility: some versions might use "Stable" not "Continue"
        if gt == "Stable":
            gt_norm = "Continue"
        else:
            gt_norm = str(gt) if gt is not None else "unknown"

        ans = t.get("answer")
        if ans == "Stable":
            ans_norm = "Continue"
        else:
            ans_norm = str(ans) if ans is not None else "unknown"

        rows.append({
            "source_file": t.get("_source_file"),
            "subject_id": t.get("subject_id"),
            "trial_index": t.get("trial_index"),
            "W": t.get("W"),
            "dataset_tag": t.get("dataset_tag"),
            "target_community": t.get("target_community"),
            "ground_truth": gt_norm,
            "answer": ans_norm,
            "correct": t.get("correct"),
            "duration_ms": t.get("duration_ms"),
            "confidence": t.get("confidence"),
            "space_toggle_count": t.get("space_toggle_count"),
            "mouse_entropy": t.get("mouse_entropy"),
            "h_alias": t.get("h_alias"),
        })

    df = pd.DataFrame(rows)

    # types
    for c in ["trial_index", "W", "target_community", "correct", "duration_ms",
              "confidence", "space_toggle_count", "mouse_entropy", "h_alias"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # normalize event labels to a clean set used in paper
    # (Your final UI options are Continue/Merge/Split/SizeChange.
    #  If you still keep "Stable" in some files, it's mapped to Continue above.)
    return df, raw_trials


def save_csv(df: pd.DataFrame, filename: str):
    path = os.path.join(TAB_DIR, filename)
    df.to_csv(path, index=False)
    print(f"[saved] {path}")


def save_txt(text: str, filename: str):
    path = os.path.join(TAB_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[saved] {path}")


def set_xticks_order(ax, labels_order: List, values_as_str: bool = True):
    if values_as_str:
        ax.set_xticks(range(len(labels_order)))
        ax.set_xticklabels([str(x) for x in labels_order], rotation=25, ha="right")
    else:
        ax.set_xticks(range(len(labels_order)))
        ax.set_xticklabels(labels_order, rotation=25, ha="right")


def plot_bar_from_series(series: pd.Series, title: str, xlabel: str, ylabel: str, out_png: str, order: List = None):
    plt.figure()
    if order is not None:
        # reindex in desired order; keep only those present
        idx = [x for x in order if x in series.index]
        series = series.reindex(idx)

    # plt.bar(series.index.astype(str), series.values.astype(float))

    # 修改样式
    plt.bar(
        series.index.astype(str),
        series.values.astype(float),
        # width=0.6,          # 👈 控制柱形宽度（默认是0.8）---W与准确性
        width=0.4,          # 👈 控制柱形宽度（默认是0.8）---W与准确性
        # color="blue",        # 填充灰色
        # edgecolor="blue",    # 边框灰色
        
        # color="#EEB7D3",        # 填充灰色
        # edgecolor="#EEB7D3",    # 边框灰色
        # alpha=0.5            # 透明度
    )
    plt.ylim(0, 1)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, out_png)
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")


def plot_hist(series: pd.Series, title: str, xlabel: str, out_png: str, bins=20):
    plt.figure()
    s = pd.to_numeric(series, errors="coerce").dropna().values
    if len(s) == 0:
        print(f"[skip] empty series for {out_png}")
        return
    plt.hist(s, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, out_png)
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")


def plot_box_by_group(df: pd.DataFrame, group_col: str, value_col: str,
                      title: str, xlabel: str, ylabel: str, out_png: str, order: List = None):
    plt.figure()
    tmp = df[[group_col, value_col]].dropna().copy()
    if tmp.empty:
        print(f"[skip] no data for {out_png}")
        return

    if order is None:
        groups = sorted(tmp[group_col].unique().tolist())
    else:
        groups = [g for g in order if g in tmp[group_col].unique()]

    data = [tmp[tmp[group_col] == g][value_col].values for g in groups]
    plt.boxplot(data, labels=[str(g) for g in groups], showfliers=False)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, out_png)
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")


# ----------------------------
# 5.1 Outputs
# ----------------------------
def make_5_1_outputs(df: pd.DataFrame):
    """
    Outputs for Section 5.1:
      - overall descriptive statistics
      - distribution plots: accuracy, RT, confidence
      - breakdown by event type and by W
    """
    # --- Basic cleaning
    df = df.copy()
    df["ground_truth"] = df["ground_truth"].astype(str)
    df["subject_id"] = df["subject_id"].astype(str)

    # Remove clearly broken rows (no ground truth / no correctness)
    df_valid = df.dropna(subset=["correct", "duration_ms", "confidence", "W", "ground_truth"]).copy()

    # Save master table for chapter 5.1
    save_csv(df_valid, "ch5_1_trials_master.csv")

    # --- Overall summary table (Table 2 candidate)
    overall = pd.DataFrame([{
        "n_subjects": df_valid["subject_id"].nunique(),
        "n_trials": len(df_valid),
        "accuracy_mean": float(df_valid["correct"].mean()),
        "rt_mean_ms": float(df_valid["duration_ms"].mean()),
        "rt_median_ms": float(df_valid["duration_ms"].median()),
        "confidence_mean": float(df_valid["confidence"].mean()),
        "confidence_median": float(df_valid["confidence"].median()),
    }])
    save_csv(overall, "table_5_1_overall_summary.csv")

    # Also save a short human-readable text snippet for easy copy into paper
    txt = (
        f"N_subjects = {overall.loc[0,'n_subjects']}\n"
        f"N_trials   = {overall.loc[0,'n_trials']}\n"
        f"Accuracy   = {overall.loc[0,'accuracy_mean']:.3f}\n"
        f"RT mean    = {overall.loc[0,'rt_mean_ms']:.1f} ms\n"
        f"RT median  = {overall.loc[0,'rt_median_ms']:.1f} ms\n"
        f"Conf mean  = {overall.loc[0,'confidence_mean']:.2f}\n"
        f"Conf median= {overall.loc[0,'confidence_median']:.2f}\n"
    )
    save_txt(txt, "table_5_1_overall_summary.txt")

    # --- By event type
    by_event = df_valid.groupby("ground_truth").agg(
        n=("correct", "count"),
        accuracy=("correct", "mean"),
        rt_mean_ms=("duration_ms", "mean"),
        rt_median_ms=("duration_ms", "median"),
        conf_mean=("confidence", "mean"),
        conf_median=("confidence", "median"),
    ).reset_index()

    # Reorder events (Continue, SizeChange, Merge, Split); tolerate unknown
    event_order = ["Continue", "SizeChange", "Merge", "Split"]
    by_event["ground_truth"] = by_event["ground_truth"].astype(str)
    by_event["order"] = by_event["ground_truth"].apply(lambda x: event_order.index(x) if x in event_order else 999)
    by_event = by_event.sort_values("order").drop(columns=["order"])
    save_csv(by_event, "table_5_1_by_event.csv")

    # --- By W
    by_W = df_valid.groupby("W").agg(
        n=("correct", "count"),
        accuracy=("correct", "mean"),
        rt_mean_ms=("duration_ms", "mean"),
        conf_mean=("confidence", "mean"),
    ).reset_index()
    by_W["W"] = by_W["W"].astype(int)
    by_W = by_W.sort_values("W")
    save_csv(by_W, "table_5_1_by_W.csv")

    # --- Optional: by subject (helps detect outliers; not necessarily in paper)
    by_subj = df_valid.groupby("subject_id").agg(
        n=("correct", "count"),
        accuracy=("correct", "mean"),
        rt_mean_ms=("duration_ms", "mean"),
        conf_mean=("confidence", "mean"),
    ).reset_index().sort_values("accuracy")
    save_csv(by_subj, "table_5_1_by_subject.csv")

    # ----------------------------
    # Figures for 5.1
    # ----------------------------

    # Fig 7 candidate: Accuracy by event type (bar)
    s_event_acc = by_event.set_index("ground_truth")["accuracy"]
    print(s_event_acc)
    plot_bar_from_series(
        s_event_acc,
        title="Accuracy by event type",
        xlabel="Event type",
        ylabel="Accuracy",
        out_png="fig_5_1_acc_by_event.png",
        order=event_order
    )

    # RT by event type (boxplot) (optional for 5.1)
    plot_box_by_group(
        df_valid, group_col="ground_truth", value_col="duration_ms",
        title="Response time by event type",
        xlabel="Event type",
        ylabel="Response time (ms)",
        out_png="fig_5_1_rt_by_event_box.png",
        order=event_order
    )

    # Accuracy by W (line-like via bar; keep it simple in 5.1)
    s_W_acc = by_W.set_index("W")["accuracy"]
    plot_bar_from_series(
        s_W_acc,
        title="Accuracy by aggregation window size",
        xlabel="W",
        ylabel="Accuracy",
        out_png="fig_5_1_acc_by_W.png",
        order=W_ORDER
    )

    # Distribution: RT and confidence (histograms)
    plot_hist(df_valid["duration_ms"], "Response time distribution", "Response time (ms)", "fig_5_1_rt_hist.png", bins=24)
    plot_hist(df_valid["confidence"], "Confidence distribution", "Confidence", "fig_5_1_conf_hist.png", bins=7)

    # Confusion matrix (often belongs to 5.3, but a quick overall version can be produced here)
    # We'll output it here as a table for convenience, you can cite later in 5.3.
    cm = pd.crosstab(df_valid["ground_truth"], df_valid["answer"])
    # ensure consistent columns/rows
    for e in event_order:
        if e not in cm.index:
            cm.loc[e] = 0
        if e not in cm.columns:
            cm[e] = 0
    cm = cm.loc[event_order, event_order]
    cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0)
    cm.to_csv(os.path.join(TAB_DIR, "table_confusion_counts.csv"))
    cm_norm.to_csv(os.path.join(TAB_DIR, "table_confusion_row_norm.csv"))
    print(f"[saved] {os.path.join(TAB_DIR, 'table_confusion_counts.csv')}")
    print(f"[saved] {os.path.join(TAB_DIR, 'table_confusion_row_norm.csv')}")

    plt.figure()
    plt.imshow(cm_norm.values, origin="upper", aspect="auto")
    plt.xticks(range(len(event_order)), event_order, rotation=25, ha="right")
    plt.yticks(range(len(event_order)), event_order)
    plt.title("Confusion matrix (row-normalized)")
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, "fig_confusion_row_norm.png")
    plt.savefig(outpath, dpi=220)
    plt.close()
    print(f"[saved] {outpath}")

    print("\n✅ Chapter 5.1 outputs complete.")
    print(f"Outputs saved to: {OUT_DIR}")


def main():
    ensure_dirs()
    df, _ = load_all_results(RESULT_DIR)
    make_5_1_outputs(df)


if __name__ == "__main__":
    main()